# Volcanic Change Detection: Reykjanes Peninsula, Iceland
### SDS210 – Programming with Spatial Data

This notebook implements a change detection workflow to identify and 
quantify landscape changes caused by the 2023–2024 Sundhnúkur eruptions 
on the Reykjanes Peninsula, Iceland.

Three research questions are investigated:
- **RQ1:** Where did the most significant landscape changes occur between 2023 and 2024?
    -> Map
- **RQ2:** How does embedding-based change detection compare to a traditional spectral approach?
- **RQ3:** Which critical infrastructure sites (Grindavík, the Blue Lagoon, and Svartsengi power station) fall within the areas of highest detected landscape change?

In [ ]:
# File paths
from pathlib import Path

# Raster reading and metadata — opening all GeoTIFFs
import rasterio
from rasterio.plot import show

# Numerical calculations
import numpy as np

# Tabular results
import pandas as pd

# Visualization — all maps and charts
import matplotlib.pyplot as plt
import cmcrameri.cm as cmc

import xarray as xr
import rioxarray

In [3]:
# --- Paths ---
DATA_RAW       = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
OUTPUT_DIR     = Path("../outputs")

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# --- Study parameters ---
AOI_NAME = "Reykjanes Peninsula, Iceland"

# Bounding box (WGS84): Sundhnúkur eruption area and Grindavík
AOI_BOUNDS = {
    "west":  -22.62,
    "east":  -22.15,
    "south":  63.80,
    "north":  63.95
}

# Temporal extent
YEAR_PRE        = 2023                       # last pre-eruption year
YEAR_POST       = 2024                       # post-eruption year

print(f"Study area:      {AOI_NAME}")
print(f"Before/after:    {YEAR_PRE} → {YEAR_POST}")

Study area:      Reykjanes Peninsula, Iceland
Baseline years:  [2017, 2018, 2019, 2020, 2021, 2022]
Before/after:    2023 → 2024


## 3. Inspection & Cleaning

### 3.1 Combine building footprints
Merging pre- and post-eruption OSM building datasets to get the most
complete picture of structures that existed before the eruptions. 


In [ ]:
# Load both raw building datasets
buildings_pre  = gpd.read_file(DATA_RAW / "buildings_pre_eruption.geojson")
buildings_post = gpd.read_file(DATA_RAW / "buildings_post_eruption.geojson")

# Check CRS of both datasets
print(f"Pre-eruption CRS:  {buildings_pre.crs}")
print(f"Post-eruption CRS: {buildings_post.crs}")

# Combine and remove duplicates by OSM feature ID
buildings_combined = pd.concat([buildings_pre, buildings_post])
buildings_combined = buildings_combined.drop_duplicates(subset=["id"])

# Convert back to GeoDataFrame
buildings_combined = gpd.GeoDataFrame(buildings_combined, crs=buildings_pre.crs)

# Save to processed folder
buildings_combined.to_file(
    DATA_PROCESSED / "buildings_combined.geojson",
    driver="GeoJSON"
)

print(f"Pre-eruption buildings:  {len(buildings_pre)}")
print(f"Post-eruption buildings: {len(buildings_post)}")
print(f"Combined (no doubles):   {len(buildings_combined)}")

# Find buildings in pre that are NOT in post
# These are candidates for destroyed structures
destroyed_candidates = buildings_pre[
    ~buildings_pre["id"].isin(buildings_post["id"])
]

print(f"Buildings that were destroyed (removed from OSM): {len(destroyed_candidates)}")
print(destroyed_candidates[["id", "geometry"]])

Pre-eruption CRS:  EPSG:4326
Post-eruption CRS: EPSG:4326
Pre-eruption buildings:  1306
Post-eruption buildings: 1383
Combined (no doubles):   1390
Buildings that were destroyed (removed from OSM): 7
                 id                                           geometry
15    way/369408544  POLYGON ((-22.4527 63.88162, -22.45262 63.8814...
550   way/414495533  POLYGON ((-22.42082 63.8486, -22.4212 63.8486,...
656   way/414495646  POLYGON ((-22.42163 63.8555, -22.42141 63.8552...
662   way/414495653  POLYGON ((-22.41732 63.84852, -22.41768 63.848...
760   way/493268696  POLYGON ((-22.43586 63.83545, -22.43568 63.835...
1168  way/812387471  POLYGON ((-22.42187 63.84846, -22.42145 63.848...
1196  way/812390811  POLYGON ((-22.4364 63.83517, -22.43618 63.8349...
